In [19]:
from langchain_core.documents import Document

In [20]:
document = Document(
    page_content="Hello",
    metadata={
        "source": "sanchit",
        "author": "sanchit"
    }
)
document

Document(metadata={'source': 'sanchit', 'author': 'sanchit'}, page_content='Hello')

In [21]:
import os
os.makedirs("../data/knowledge/", exist_ok=True)

In [22]:
file_sample = {
    "../data/knowledge/python.txt": "Hello from sanchit"
}

for file_path, content in file_sample.items():
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)

### Text Loader

In [23]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/knowledge/python.txt", encoding='utf-8')
document = loader.load()
print(document)

[Document(metadata={'source': '../data/knowledge/python.txt'}, page_content='Hello from sanchit')]


### Directory loader for pdf

In [24]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader

loader = DirectoryLoader(
    "../data/knowledge/pdf/",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False
)

document = loader.load()
print(len(document))

2


### Chunking

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(document, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["/n", " ", ""]
    )
    split_docs = text_splitter.split_documents(document)
    print(f"split {len(document)} documents into {len(split_docs)} chunks")

    print(f"\n Example chunk")
    print(f"content: {split_docs[0].page_content[:200]}")
    print(f"metadata: {split_docs[0].metadata}")

    return split_docs

chunks = split_documents(document)
chunks

split 2 documents into 9 chunks

 Example chunk
content: Department of Training and Placement, NIT Trichy 620015 
Telephone : +91-431-2501081    e-mail: tp@nitt.edu, tnp.nitt@gmail.com 
 
 
 
Educational Qualification
Academic Achievements 
• 
Achieved Pupi
metadata: {'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-22T14:30:33+05:30', 'source': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'file_path': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Charvini Doddi', 'subject': '', 'keywords': '', 'moddate': '2026-08-22T14:30:33+05:30', 'trapped': '', 'modDate': "D:20260822143033+05'30'", 'creationDate': "D:20260822143033+05'30'", 'page': 0}


[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-08-22T14:30:33+05:30', 'source': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'file_path': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'total_pages': 2, 'format': 'PDF 1.7', 'title': '', 'author': 'Charvini Doddi', 'subject': '', 'keywords': '', 'moddate': '2026-08-22T14:30:33+05:30', 'trapped': '', 'modDate': "D:20260822143033+05'30'", 'creationDate': "D:20260822143033+05'30'", 'page': 0}, page_content='Department of Training and Placement, NIT Trichy 620015 \nTelephone : +91-431-2501081    e-mail: tp@nitt.edu, tnp.nitt@gmail.com \n \n \n \nEducational Qualification\nAcademic Achievements \n• \nAchieved Pupil rank on Codeforces. \n• \nSolved 1,000+ problems on LeetCode with a rating of 1788.  \n \nInternship Experience \nSAP Labs, SDE Intern – Joule                                                                                  

### Embedding and vector store

In [26]:
from typing import List
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document


class EmbeddingManager:

    def __init__(self):
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

    def embed_documents(
        self,
        documents: List[Document]
    ) -> List[List[float]]:
        texts = [document.page_content for document in documents]
        return self.embedding_model.embed_documents(texts)

    def embed_query(
        self,
        query: str
    ) -> List[float]:
        return self.embedding_model.embed_query(query)


embedding_manager = EmbeddingManager()

vectors = embedding_manager.embed_documents(chunks)

print(len(vectors))
print(len(vectors[0]))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2418.89it/s]


9
384


### vector store

In [27]:
import chromadb
from typing import List, cast
from chromadb.api.types import Embedding, Metadata
from langchain_core.documents import Document


class VectorStore:

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name
        )

    def add_documents(
        self,
        documents: List[Document],
        embeddings: List[List[float]]
    ) -> None:

        ids = [f"doc_{i}" for i in range(len(documents))]

        texts = [
            document.page_content
            for document in documents
        ]

        metadatas = [
            document.metadata
            for document in documents
        ]

        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=cast(List[Embedding], embeddings),
            metadatas=cast(List[Metadata], metadatas)
        )

    def similarity_search(
    self,
    query_embedding: List[float],
    k: int = 5
    ) -> List[Document]:

        results = self.collection.query(
            query_embeddings=[
                cast(Embedding, query_embedding)
            ],
            n_results=k
        )

        documents = results["documents"]
        metadatas = results["metadatas"]

        if documents is None:
            return []

        if metadatas is None:
            metadatas = [[] for _ in documents]

        return [
            Document(
                page_content=text,
                metadata=metadata or {}
            )
            for text, metadata in zip(documents[0], metadatas[0])
        ]

vector_store = VectorStore()
vector_store.add_documents(chunks, vectors)

### Retrieval

In [28]:
query = "What is the name of the candidate in the resume"

query_embedding = embedding_manager.embed_query(query)

retrieved_documents = vector_store.similarity_search(query_embedding, 2);
print(retrieved_documents)

[Document(metadata={'title': '', 'subject': '', 'keywords': '', 'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'page': 1, 'creationDate': "D:20260822143033+05'30'", 'format': 'PDF 1.7', 'moddate': '2026-08-22T14:30:33+05:30', 'creationdate': '2026-08-22T14:30:33+05:30', 'trapped': '', 'source': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'file_path': '../data/knowledge/pdf/110123099_sanchitRathore.pdf', 'author': 'Charvini Doddi', 'modDate': "D:20260822143033+05'30'", 'total_pages': 2}, page_content='Canvas, CSS, Tailwind CSS \n• \nFrameworks and Libraries       : Spring Boot, React.js, Next.js, Node.js, Express.js  \n• \nDeveloper Tools                       : Git, Postman, Docker, Kubernetes \n• \nDatabase                                   : MongoDB, MySQL, PostgreSQL \n• \nOther                                         : Linux(Basics), OS, DBMS, OOP \nPositions of Responsibility \n• \nSoftware Developer, Delta Force Club:   